In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn

from src.data.load_data import load_data

In [12]:
PROJECT_ROOT

WindowsPath('c:/Users/pomia/Projects/customer-intelligence-ml-platform')

In [13]:
mlflow.set_tracking_uri(
    "http://127.0.0.1:5000"
)

mlflow.set_experiment(
    "customer-churn"
)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1786050839413, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786050839413, lifecycle_stage='active', name='customer-churn', tags={}, trace_location=None, workspace='default'>

In [14]:
df = load_data()

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [15]:
X = df.drop(
    columns=["Churn", "customerID"]
)

y = df["Churn"].map(
    {
        "No": 0,
        "Yes": 1
    }
)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [17]:
from src.pipelines.churn_pipeline import create_pipeline
from src.evaluation.evaluate import evaluate_model

In [18]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

categorical_features = [
    col
    for col in X.columns
    if col not in numeric_features
]

In [19]:
clf = create_pipeline(
    model_name="logistic_regression",
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

In [20]:
with mlflow.start_run(
    run_name="logistic-regression-training"
):

    clf.fit(
        X_train,
        y_train,
    )

    metrics = evaluate_model(
        clf,
        X_test,
        y_test,
    )

    mlflow.log_param(
        "model",
        "Logistic Regression",
    )

    for name, value in metrics.items():
        mlflow.log_metric(
            name,
            value,
        )

    mlflow.sklearn.log_model(
        clf,
        "model",
        serialization_format="cloudpickle",
    )

    print(metrics)

2026/08/07 00:03:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/07 00:03:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/07 00:03:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'accuracy': 0.8055358410220014, 'precision': 0.6572327044025157, 'recall': 0.5588235294117647, 'f1': 0.6040462427745664, 'roc_auc': 0.8418610659019866}
🏃 View run logistic-regression-training at: http://127.0.0.1:5000/#/experiments/1/runs/8d7351c29841428da3df8318fd957191
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
